# Visualização da Geometria do Arranjo de Sensores

Este notebook demonstra a geração e visualização de diferentes geometrias de arranjos utilizados em simulações de DOA.

- Linear
- Circular
- Planar
- Random
- Esférico
- Cúbico


## Imports iniciais

In [13]:
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from enum import Enum
from typing import Dict

## Definição de tipos necessários

In [14]:
class GeometryType(Enum):
    """Enum class for array geometry types."""
    LINEAR = "linear"
    CIRCULAR = "circular"
    PLANAR = "planar"
    RANDOM = "random"
    SPHERICAL = "spherical"
    CUBIC = "cubic"


## 📌 Função auxiliar para criar a geometria

In [15]:
def create_geometry(num_sensors, geometry_type, spacing=.5, wavelength=1.0):
    params: Dict = {
        "num_sensors": num_sensors,
        "num_snapshots": 100,
        "geometry": geometry_type,
        "array_elements_spacing": spacing,
        "wavelength": wavelength,
    }
    
    return params
    
def generate_array_structure(array_params: Dict) -> np.ndarray:
    """
    Generate array geometry parameters based on the provided configuration.
    Array positions tensor:
    - Shape: (N, D)
        where   N = num_sensors
                D = 1 (LINEAR), 2 (PLANAR), 3 (CUBIC/SPHERICAL)

    Example:
    - Linear: [[0], [d], [2d], ...]
        where d is the distance between sensors
    - Planar: [[x0, y0], [x1, y1], ...]
        where x0, y0 are the coordinates of the sensor in a planar array
    - Circular: [[x0, y0], [x1, y1], ...]
        where x0, y0 are the coordinates of the sensor in a circular array
    - Cubic:  [[x0, y0, z0], [x1, y1, z1], ...]
        where x0, y0, z0 are the coordinates of the sensor in a cubic array
    - Spherical: [[r0, theta0, phi0], [r1, theta1, phi1], ...]
        where r0, theta0, phi0 are the spherical coordinates of the sensor
    """

    # Generate sensor positions based on the geometry type
    match array_params["geometry"]:
        case GeometryType.LINEAR:
            # Linear array
            # Create a linear array with spacing between sensors and reshape it to (num_sensors, 1)
            # in order to avoid unidimensional array (num_sensors,) vs. (num_sensors, 1)
            # and to make it compatible with the rest of the code
            inter_element_distance = 0.5 * array_params["wavelength"]
            array_structure = (np.arange(array_params["num_sensors"]) * inter_element_distance).reshape(array_params["num_sensors"], 1)
        case GeometryType.CIRCULAR:
            # Circular array
            angles = np.linspace(0, 2 * np.pi, array_params["num_sensors"], endpoint=False)
            array_structure = np.column_stack((np.cos(angles), np.sin(angles)))
        case GeometryType.PLANAR:
            # Planar array
            x_positions = np.arange(array_params["num_sensors"]) * array_params["array_elements_spacing"]
            y_positions = np.arange(array_params["num_sensors"]) * array_params["array_elements_spacing"]
            array_structure = np.array(np.meshgrid(x_positions, y_positions)).T.reshape(-1, 2)
        case GeometryType.RANDOM:
            # Random array
            array_structure = np.random.rand(array_params["num_sensors"], 2) * array_params["array_elements_spacing"]
        case GeometryType.SPHERICAL:
            # Spherical array
            # Generate spherical coordinates
            # r: radius (constant for spherical array)
            # theta: polar angle (0 to pi)
            # phi: azimuthal angle (0 to 2*pi)
            r = 1  # Radius of the sphere
            theta = np.linspace(0, np.pi, array_params["num_sensors"])
            phi = np.linspace(0, 2 * np.pi, array_params["num_sensors"])
            theta, phi = np.meshgrid(theta, phi)
            theta = theta.flatten()
            phi = phi.flatten()
            x = r * np.sin(theta) * np.cos(phi)
            y = r * np.sin(theta) * np.sin(phi)
            z = r * np.cos(theta)
            array_structure = np.column_stack((x, y, z))
        case GeometryType.CUBIC:
            # Cubic array
            x_positions = np.arange(array_params["num_sensors"]) * array_params["array_elements_spacing"]
            y_positions = np.arange(array_params["num_sensors"]) * array_params["array_elements_spacing"]
            z_positions = np.arange(array_params["num_sensors"]) * array_params["array_elements_spacing"]
            array_structure = np.array(np.meshgrid(x_positions, y_positions, z_positions)).T.reshape(-1, 3)

    return array_structure

## 📌 Função para plotar com Plotly

In [16]:
def plot_array_plotly(positions, geometry_name):
    dim = positions.shape[1]
    fig = go.Figure()

    if dim == 1:
        fig.add_trace(go.Scatter(x=positions[:, 0], y=[0]*len(positions),
                                 mode='markers', marker=dict(size=8)))
        fig.update_layout(title=f"Linear Array", xaxis_title="X", yaxis_title="")
        
    elif dim == 2:
        fig.add_trace(go.Scatter(x=positions[:, 0], y=positions[:, 1],
                                 mode='markers', marker=dict(size=8)))
        fig.update_layout(title=f"{geometry_name} Array", xaxis_title="X", yaxis_title="Y")
        
    elif dim == 3:
        fig.add_trace(go.Scatter3d(
            x=positions[:, 0], y=positions[:, 1], z=positions[:, 2],
            mode='markers', marker=dict(size=5)
        ))
        fig.update_layout(
            title=f"{geometry_name} Array",
            scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z')
        )
        
    fig.show()
    return fig


## 📌 Função para exportar imagem

In [17]:
def export_figure(fig, filename_base):
    fig.write_image(f"{filename_base}.pdf")
    fig.write_image(f"{filename_base}.png")
    print(f"Exported: {filename_base}.pdf and {filename_base}.png")


## 🚀 Execução: visualizando as geometrias

In [18]:
geometries = [
    GeometryType.LINEAR,
    GeometryType.CIRCULAR,
    GeometryType.PLANAR,
    GeometryType.RANDOM,
    GeometryType.SPHERICAL,
    GeometryType.CUBIC
]

for geo in geometries:
    print(f"Generating {geo.value} array...")
    positions = generate_array_structure(create_geometry(num_sensors=8, geometry_type=geo))
    fig = plot_array_plotly(positions, geometry_name=geo.name)
    export_figure(fig, f"output/array_{geo.name.lower()}")


Generating linear array...


Exported: output/array_linear.pdf and output/array_linear.png
Generating circular array...


Exported: output/array_circular.pdf and output/array_circular.png
Generating planar array...


Exported: output/array_planar.pdf and output/array_planar.png
Generating random array...


Exported: output/array_random.pdf and output/array_random.png
Generating spherical array...


Exported: output/array_spherical.pdf and output/array_spherical.png
Generating cubic array...


Exported: output/array_cubic.pdf and output/array_cubic.png
